# 02 — Fetch Author Metrics from Google Scholar via SerpAPI

This notebook shows how to:
1. Retrieve an author's full Google Scholar profile using their author ID
2. Extract key metrics: h-index, i10-index, total citations
3. List their publications with citation counts

**API docs:** https://serpapi.com/google-scholar-author-api

In [7]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from serpapi import GoogleSearch

In [8]:
load_dotenv('../.env')
SERPAPI_KEY = os.getenv('SERPAPI_KEY')
if not SERPAPI_KEY or SERPAPI_KEY == 'your_serpapi_key_here':
    raise ValueError('SERPAPI_KEY not set.')

## 1. Fetch the author profile

Use a Google Scholar author ID (found in notebook 01 or from a Google Scholar profile URL).

In [9]:
# Example: Rasheed Omobolaji Alabi
# You can find this ID in the URL of a Google Scholar profile page:
# https://scholar.google.com/citations?user=<AUTHOR_ID>
AUTHOR_ID = 'AK9BE00AAAAJ'  # replace with your target author's ID

params = {
    'engine': 'google_scholar_author',
    'author_id': AUTHOR_ID,
    'api_key': SERPAPI_KEY,
}

search = GoogleSearch(params)
author_data = search.get_dict()

print('Author name:', author_data.get('author', {}).get('name'))
print('Affiliation:', author_data.get('author', {}).get('affiliations'))

Author name: Lukas Liss
Affiliation: PhD Student, Process and Data Science, Computer Science, RWTH Aachen University


## 2. Extract citation metrics

In [10]:
# The 'cited_by' key contains a table with h-index and i10-index
cited_by = author_data.get('cited_by', {})
table = cited_by.get('table', [])

print('Citation metrics:')
for row in table:
    for metric, values in row.items():
        print(f"  {metric:20s}: all={values.get('all', 'N/A'):>6}  since 2019={values.get('since_2019', 'N/A'):>6}")

Citation metrics:
  citations           : all=   146  since 2019=   N/A
  h_index             : all=     6  since 2019=   N/A
  i10_index           : all=     5  since 2019=   N/A


In [11]:
# Convenience extraction
def extract_author_metrics(author_data: dict) -> dict:
    author_info = author_data.get('author', {})
    cited_by = author_data.get('cited_by', {})
    table = cited_by.get('table', [])

    h_index = i10_index = total_citations = None
    for row in table:
        if 'citations' in row:
            total_citations = row['citations'].get('all')
        if 'h_index' in row:
            h_index = row['h_index'].get('all')
        if 'i10_index' in row:
            i10_index = row['i10_index'].get('all')

    return {
        'name': author_info.get('name'),
        'affiliations': author_info.get('affiliations'),
        'scholar_id': author_info.get('author_id'),
        'total_citations': total_citations,
        'h_index': h_index,
        'i10_index': i10_index,
    }


metrics = extract_author_metrics(author_data)
for k, v in metrics.items():
    print(f'{k:20s}: {v}')

name                : Lukas Liss
affiliations        : PhD Student, Process and Data Science, Computer Science, RWTH Aachen University
scholar_id          : None
total_citations     : 146
h_index             : 6
i10_index           : 5


## 3. List publications (articles)

In [12]:
articles = author_data.get('articles', [])
print(f'Articles returned on this page: {len(articles)}')

df_articles = pd.DataFrame([
    {
        'title': a.get('title', '')[:80],
        'year': a.get('year'),
        'cited_by': a.get('cited_by', {}).get('value', 0),
        'authors': a.get('authors', '')[:60],
    }
    for a in articles
])

df_articles.sort_values('cited_by', ascending=False)

Articles returned on this page: 15


,title,year,cited_by,authors
0,OCEL (object-centric event log) 2.0 specification,2024,52.0,"A Berti, I Koren, JN Adams, G Park, B Knopp, N..."
1,Object-centric alignments,2023,37.0,"L Liss, JN Adams, WMP van der Aalst"
2,Ocel (object-centric event log) 2.0 specificat...,,12.0,"A Berti, I Koren, JN Adams, G Park, B Knopp, N..."
3,A systematic review of machine learning-based ...,2025,11.0,"B Critelli, A Hassan, I Lahooti, L Noh, JS Par..."
4,Totem: Temporal object type model for object-c...,2024,10.0,"L Liss, JN Adams, WMP van der Aalst"
5,Towards a Catalog of Refactoring Solutions for...,2021,8.0,"L Liss, H Kämmerling, P Alexander, H Lichter"
6,ImPres: an immersive 3D presentation framework...,2021,5.0,"B Hensen, L Liß, R Klamma"
7,Framework for extracting real-world object-cen...,2024,3.0,"L Liss, N Elbert, CM Flath, WMP van der Aalst"
8,Critical appraisal of machine learning prognos...,2024,3.0,"A Hassan, B Critelli, I Lahooti, A Lahooti, N ..."
9,Object-centric causal nets,2025,2.0,"L Liss, C Mensing, WMP van der Aalst"


## 4. Paginate through all articles

By default SerpAPI returns 20 articles. Use the `start` parameter to paginate.

In [13]:
import time

def get_all_articles(author_id: str, api_key: str) -> list:
    """Retrieve all articles for a Google Scholar author, handling pagination."""
    all_articles = []
    start = 0
    page_size = 100

    while True:
        params = {
            'engine': 'google_scholar_author',
            'author_id': author_id,
            'sort': 'pubdate',
            'num': page_size,
            'start': start,
            'api_key': api_key,
        }
        results = GoogleSearch(params).get_dict()
        page = results.get('articles', [])

        if not page:
            break
        all_articles.extend(page)

        # If fewer articles than requested were returned, we've reached the end
        if len(page) < page_size:
            break
        start += page_size
        time.sleep(0.5)

    print(f'Total articles fetched: {len(all_articles)}')
    return all_articles


all_articles = get_all_articles(AUTHOR_ID, SERPAPI_KEY)

Total articles fetched: 15


In [14]:
# Convert to DataFrame and show publication years
df_all = pd.DataFrame([
    {'title': a.get('title', ''), 'year': a.get('year'), 'cited_by': a.get('cited_by', {}).get('value', 0)}
    for a in all_articles
])

print('Publications per year:')
print(df_all.groupby('year').size().sort_index())

Publications per year:
year
        1
2021    3
2023    1
2024    6
2025    4
dtype: int64
